# Canal binario simétrico

Este cuaderno acompaña los temas de canales de comunicación e información mutua.

Un canal binario simétrico recibe bits y, con probabilidad `p`, invierte cada bit. Si `p = 0`, el canal no tiene ruido. Si `p = 0.5`, la salida es tan ruidosa como lanzar una moneda.

## Simulación del canal

Primero definimos una función que transmite una secuencia de bits con una probabilidad de error controlada.

In [ ]:
import math
import random


def transmitir(bits, probabilidad_error, semilla=7):
    if not 0 <= probabilidad_error <= 1:
        raise ValueError("La probabilidad de error debe estar entre 0 y 1.")
    rng = random.Random(semilla)
    salida = []
    for bit in bits:
        if bit not in (0, 1):
            raise ValueError("El mensaje solo puede contener bits 0 o 1.")
        hay_error = rng.random() < probabilidad_error
        salida.append(1 - bit if hay_error else bit)
    return salida


def tasa_error(entrada, salida):
    errores = sum(a != b for a, b in zip(entrada, salida))
    return errores / len(entrada)

## Un mensaje pequeño

Transmitimos una secuencia corta para ver cómo aparecen errores bit a bit.

In [ ]:
mensaje = [1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1]
recibido = transmitir(mensaje, probabilidad_error=0.15, semilla=2)

print("entrada:", mensaje)
print("salida: ", recibido)
print(f"tasa de error observada: {tasa_error(mensaje, recibido):.2%}")

## Entropía binaria y capacidad

Para un canal binario simétrico con probabilidad de error `p`, la capacidad es:

```text
C = 1 - H_b(p)
```

donde `H_b(p)` es la entropía de una moneda con probabilidades `p` y `1-p`.

In [ ]:
def entropia_binaria(p):
    if p in (0, 1):
        return 0.0
    return -p * math.log2(p) - (1 - p) * math.log2(1 - p)


def capacidad_canal_binario(p):
    return 1 - entropia_binaria(p)


for p in [0, 0.01, 0.05, 0.1, 0.2, 0.5]:
    print(f"p = {p:>4.2f}  H_b(p) = {entropia_binaria(p):.4f}  C = {capacidad_canal_binario(p):.4f}")

## Simulación con muchos bits

Con mensajes largos, la tasa de error observada se acerca a la probabilidad de error configurada.

In [ ]:
rng = random.Random(11)
mensaje_largo = [rng.randrange(2) for _ in range(10_000)]

for p in [0.01, 0.05, 0.1, 0.2]:
    recibido = transmitir(mensaje_largo, p, semilla=23)
    print(f"p esperado = {p:.2%}  tasa observada = {tasa_error(mensaje_largo, recibido):.2%}")

In [ ]:
probabilidades = [i / 100 for i in range(0, 51)]
capacidades = [capacidad_canal_binario(p) for p in probabilidades]

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib no está instalado; se omite la gráfica.")
else:
    plt.figure(figsize=(7, 4))
    plt.plot(probabilidades, capacidades)
    plt.xlabel("Probabilidad de error p")
    plt.ylabel("Capacidad en bits por uso")
    plt.title("Capacidad del canal binario simétrico")
    plt.grid(True, alpha=0.3)
    plt.show()

## Para experimentar

1. Cambia la longitud de `mensaje_largo` y observa cómo varía la tasa de error.
2. Prueba `p = 0.5` y explica por qué la capacidad cae a cero.
3. Modifica `transmitir` para que el canal no sea simétrico: una probabilidad para cambiar `0 -> 1` y otra para `1 -> 0`.